# 🚗 TakarMobil: Data Preprocessing & Pipeline Preparation
*TakarMobil: Sistem Estimasi Rentang Harga & Analisis Faktor Penentu Harga Mobil Bekas*

---

### 🎯 Tujuan Tahap Preprocessing (Phase 3):
1. **Data Cleaning & Parsing:** Menghapus data tanpa harga, filter harga wajar (Rp 25 Jt–10 M), filter 5 merek utama, parsing tahun (1995–2025), serta perbaikan parsing kilometer (menangani format `K` ribuan dari Mobil123).
2. **Deduplikasi Penuh:** Menghapus baris ganda berdasarkan `URL` dan iklan repost identik `(Judul, Merek, Tahun, Harga, Lokasi)` guna mencegah *data leakage*.
3. **Handling Missing Values:** Imputasi nilai kosong pada `Kilometer` menggunakan median bertingkat (`Model + Tahun` → `Merek + Tahun` → Global).
4. **Feature Engineering:** Membentuk `Usia_Mobil`, `Wilayah`, `Merek_Model`, dan target $\log(1 + \text{Harga})$.
5. **Skema Pembagian Data:** **`StratifiedShuffleSplit` (80% Train : 20% Test)** berbasis stratifikasi merek/tipe agar seluruh tipe mobil populer terwakili tanpa kebocoran data.
6. **Pipeline Preprocessing:** `ColumnTransformer` (`StandardScaler` + `OneHotEncoder` termasuk `Model_Clean`).

## 1. Import Library & Setup Lingkungan
Memuat pustaka standar untuk manipulasi data, scikit-learn untuk *preprocessing pipeline* dan *data splitting*, serta joblib untuk penyimpanan artifak preprocessor.


In [1]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
print("Pustaka preprocessing berhasil dimuat!")

Pustaka preprocessing berhasil dimuat!


## 2. Pemuatan Dataset Mentah (Raw Data)
Memuat dataset mentah `Data Scraping.xlsx` menggunakan deteksi path dinamis yang fleksibel.


In [2]:
candidate_paths = [
    "data/raw/Data Scraping.xlsx",
    "../data/raw/Data Scraping.xlsx",
    "Data Scraping.xlsx"
]

file_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if file_path is None:
    raise FileNotFoundError("File Data Scraping.xlsx tidak ditemukan!")

print(f"Memuat dataset mentah dari: {file_path}")
df_raw = pd.read_excel(file_path)
print(f"Jumlah baris awal: {df_raw.shape[0]}, Jumlah kolom: {df_raw.shape[1]}")
df_raw.head(3)


Memuat dataset mentah dari: ../data/raw/Data Scraping.xlsx


Jumlah baris awal: 17670, Jumlah kolom: 16


,Sumber,URL,Judul,Merek,Model,Varian,Tahun,Kilometer,Transmisi,Warna,Kapasitas Mesin,Bahan Bakar,Tanggal Upload,Lokasi,Tipe Bodi,Harga
0,OLX,https://www.olx.co.id/item/939502279,MERCEDES BENZ A45 AMG 2014 2.0 TURBO AWD PANOR...,Mercedes-Benz,A45 AMG,2.0 Bensin-AT,2014,NaN,Automatic,Putih,NaN,Bensin,07-10-2025,Jakarta Timur,NaN,499.000.000
1,OLX,https://www.olx.co.id/item/939871468,Mercedes Benz Sprinter A415 CDI 2022,Mercedes-Benz,Sprinter,NaN,2021,NaN,Manual,Hitam,NaN,Diesel,17-10-2025,Jakarta Utara,NaN,1.850.000.000
2,OLX,https://www.olx.co.id/item/939450820,[ GARANSI 5TH ] Mercedes Benz Mercy GLC200 GLC...,Mercedes-Benz,GLC200,2.0 AMG Line Bensin-AT,2022,NaN,Automatic,Putih,NaN,Bensin,06-10-2025,Jakarta Barat,NaN,720.000.000


## 3. Data Cleaning & Standarisasi Fitur
Pada tahap ini kita menyelesaikan berbagai anomali teks dan artefak web scraping yang telah ditemukan pada tahap EDA:
1. **Drop Kolom Tak Relevan:** Kolom `URL`, `Judul`, `Tanggal Upload`, dan `Tipe Bodi` (karena >47% kosong) dihapus dari fitur pemodelan.
2. **Parsing Harga:** Mengonversi string format Rupiah (bertitik) menjadi float murni, lalu menghapus data tanpa harga.
3. **Penyaringan Outlier Ekstrem:** Membatasi harga mobil bekas pada rentang rasional (Rp 25 Juta s/d Rp 10 Miliar).
4. **Standarisasi Merek:** Memperbaiki kesalahan scraping di mana tipe seri Mercedes (seperti `C-Class`, `E-Class`, `GLC-Class`, `GLA-Class`) terdata sebagai merek terpisah.
5. **Penyatuan Bahan Bakar:** Menyatukan label `Bensin` dan `Petrol` (sinonim dari dua platform scraping berbeda) menjadi satu kategori: `Bensin`.
6. **Parsing Kilometer & Tahun:** Mengambil nilai tengah dari rentang teks kilometer dan memvalidasi rentang tahun kendaraan.


In [3]:
df = df_raw.copy()

# A. Drop Baris tanpa Harga & Parsing Harga
def clean_price(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    digits = ''.join(c for c in str(val).replace('.', '').strip() if c.isdigit())
    return float(digits) if digits else np.nan

df['Harga_Num'] = df['Harga'].apply(clean_price)
df = df.dropna(subset=['Harga_Num'])

# Filter rentang harga rasional (25 Juta - 10 Miliar)
df = df[(df['Harga_Num'] >= 25_000_000) & (df['Harga_Num'] <= 10_000_000_000)]
print(f"Jumlah data setelah pembersihan & filter harga: {len(df)} baris")


Jumlah data setelah pembersihan & filter harga: 17363 baris


In [4]:
# B. Standarisasi Kolom Merek
# Kamus pemetaan seri mobil Mercedes-Benz yang terpisah saat scraping
mercedes_series = [
    'C-Class', 'E-Class', 'GLC-Class', 'GLA-Class', 'CLA-Class', 'GLE-Class',
    'A-Class', 'S-Class', 'GLS-Class', 'V-Class', 'CLS-Class', 'SLK-Class',
    'GLB-Class', 'B-Class', 'ML-Class', 'G-Class', 'GL-Class', 'CLS400',
    'CLA200', 'SL-Class', 'Mercedes Benz'
]

# Fungsi standarisasi merek
def standardize_brand(m):
    if pd.isna(m):
        return 'Lainnya'
    m_str = str(m).strip()
    if m_str in mercedes_series or 'mercedes' in m_str.lower() or 'benz' in m_str.lower():
        return 'Mercedes-Benz'
    if 'toyota' in m_str.lower():
        return 'Toyota'
    if 'daihatsu' in m_str.lower():
        return 'Daihatsu'
    if 'suzuki' in m_str.lower():
        return 'Suzuki'
    if 'honda' in m_str.lower():
        return 'Honda'
    return m_str

df['Merek_Clean'] = df['Merek'].apply(standardize_brand)

# Filter hanya 5 Merek Utama Proyek
TARGET_BRANDS = ['Toyota', 'Daihatsu', 'Suzuki', 'Honda', 'Mercedes-Benz']
df = df[df['Merek_Clean'].isin(TARGET_BRANDS)]

print("Distribusi 5 Merek Utama setelah standarisasi:")
print(df['Merek_Clean'].value_counts())


Distribusi 5 Merek Utama setelah standarisasi:
Merek_Clean
Toyota           3748
Daihatsu         3444
Mercedes-Benz    3347
Suzuki           3222
Honda            3079
Name: count, dtype: int64


In [5]:
# C. Standarisasi Transmisi
def clean_transmission(val):
    if pd.isna(val): return 'Automatic'
    v = str(val).lower()
    if 'auto' in v or 'at' in v: return 'Automatic'
    if 'man' in v or 'mt' in v: return 'Manual'
    return 'Automatic'
df['Transmisi_Clean'] = df['Transmisi'].apply(clean_transmission)

# D. Standarisasi Bahan Bakar
def clean_fuel(val):
    if pd.isna(val): return 'Bensin'
    v = str(val).lower()
    if 'bensin' in v or 'petrol' in v or 'gasoline' in v: return 'Bensin'
    if 'diesel' in v or 'solar' in v: return 'Diesel'
    if 'hybrid' in v or 'listrik' in v or 'electric' in v: return 'Hybrid'
    return 'Bensin'
df['Bahan_Bakar_Clean'] = df['Bahan Bakar'].apply(clean_fuel)

# E. Parsing Kilometer (menangani format '10 - 15K km' dari Mobil123 & '50.000-55.000 km' dari OLX)
def clean_km(val):
    if pd.isna(val): return np.nan
    s = str(val).lower().replace('km', '').strip()
    is_k = 'k' in s
    s = s.replace('k', '').strip()
    nums = []
    for p in s.split('-'):
        d = ''.join(c for c in p.replace('.', '').replace(',', '').strip() if c.isdigit())
        if d:
            v = float(d)
            if is_k or v <= 500:
                v *= 1000.0
            nums.append(v)
    if not nums: return np.nan
    res = float(np.mean(nums))
    return res if 0 <= res <= 400_000 else np.nan
df['KM_Num'] = df['Kilometer'].apply(clean_km)

# F. Parsing Tahun
def clean_year(val):
    if pd.isna(val): return np.nan
    d = ''.join(c for c in str(val) if c.isdigit())
    if d and len(d) == 4:
        y = int(d)
        if 1995 <= y <= 2025: return y
    return np.nan
df['Tahun_Num'] = df['Tahun'].apply(clean_year)
df = df.dropna(subset=['Tahun_Num'])
df['Tahun_Num'] = df['Tahun_Num'].astype(int)

# G. Standarisasi Model Mobil
df['Model_Clean'] = df['Model'].fillna('Lainnya').astype(str).str.strip().str.title()

# H. Simplifikasi Wilayah Lokasi
def simplify_location(loc):
    if pd.isna(loc): return 'Jakarta'
    l = str(loc).lower()
    if 'jakarta' in l: return 'Jakarta'
    elif 'bogor' in l: return 'Bogor'
    elif 'depok' in l: return 'Depok'
    elif 'tangerang' in l: return 'Tangerang'
    elif 'bekasi' in l: return 'Bekasi'
    else: return 'Luar Jabodetabek'
df['Wilayah'] = df['Lokasi'].apply(simplify_location)

# I. Deduplikasi Data Ganda (URL & Iklan Repost Identik)
before_dedup = len(df)
df = df.drop_duplicates(subset=['URL']).reset_index(drop=True)
after_url = len(df)
df = df.drop_duplicates(subset=['Judul', 'Merek_Clean', 'Tahun_Num', 'Harga_Num', 'Lokasi']).reset_index(drop=True)

print(f"Sebelum deduplikasi : {before_dedup} baris")
print(f"Setelah hapus URL ganda : {after_url} baris (-{before_dedup - after_url})")
print(f"Setelah hapus iklan identik (Judul+Merek+Tahun+Harga+Lokasi): {len(df)} baris (-{after_url - len(df)})")
df[['Merek_Clean', 'Model_Clean', 'Tahun_Num', 'KM_Num', 'Transmisi_Clean', 'Bahan_Bakar_Clean', 'Wilayah', 'Harga_Num']].head()

Sebelum deduplikasi : 16799 baris
Setelah hapus URL ganda : 14754 baris (-2045)
Setelah hapus iklan identik (Judul+Merek+Tahun+Harga+Lokasi): 13359 baris (-1395)


,Merek_Clean,Model_Clean,Tahun_Num,KM_Num,Transmisi_Clean,Bahan_Bakar_Clean,Wilayah,Harga_Num
0,Mercedes-Benz,A45 Amg,2014,NaN,Automatic,Bensin,Jakarta,4.990000e+08
1,Mercedes-Benz,Sprinter,2021,NaN,Manual,Diesel,Jakarta,1.850000e+09
2,Mercedes-Benz,Glc200,2022,NaN,Automatic,Bensin,Jakarta,7.200000e+08
3,Mercedes-Benz,Gle400,2016,NaN,Automatic,Bensin,Jakarta,5.980000e+08
4,Mercedes-Benz,Lainnya,2010,NaN,Automatic,Bensin,Luar Jabodetabek,1.590000e+08


## 4. Penanganan Missing Values (Imputasi Cerdas)
Kolom `KM_Num` memiliki nilai kosong sekitar 20%. Kita tidak melakukan imputasi global yang sederhana (mean seluruh dataset), melainkan menggunakan **Imputasi Hierarkis Bertingkat**:
1. Tingkat 1: Nilai median jarak tempuh berdasarkan kombinasi `Model_Clean` + `Tahun_Num`.
2. Tingkat 2 (Fallback): Jika kombinasi model & tahun tersebut tidak ada, gunakan median berdasarkan `Merek_Clean` + `Tahun_Num`.
3. Tingkat 3 (Global Fallback): Median seluruh dataset.


In [6]:
print(f"Jumlah missing KM sebelum imputasi: {df['KM_Num'].isnull().sum()}")

# 1. Median berdasarkan Model + Tahun
km_imputer_model_year = df.groupby(['Model_Clean', 'Tahun_Num'])['KM_Num'].transform('median')
df['KM_Clean'] = df['KM_Num'].fillna(km_imputer_model_year)

# 2. Median berdasarkan Merek + Tahun (Fallback jika kelompok model terlalu sedikit)
km_imputer_brand_year = df.groupby(['Merek_Clean', 'Tahun_Num'])['KM_Num'].transform('median')
df['KM_Clean'] = df['KM_Clean'].fillna(km_imputer_brand_year)

# 3. Median global (Fallback terakhir)
df['KM_Clean'] = df['KM_Clean'].fillna(df['KM_Clean'].median())

print(f"Jumlah missing KM setelah imputasi: {df['KM_Clean'].isnull().sum()}")


Jumlah missing KM sebelum imputasi: 2933
Jumlah missing KM setelah imputasi: 0


## 5. Feature Engineering
Membentuk fitur-fitur baru yang bernilai prediktif tinggi terhadap harga mobil bekas:
1. **`Usia_Mobil`:** Selisih tahun acuan (2025) dengan tahun perakitan mobil ($2025 - \text{Tahun}$).
2. **`Log_Harga`:** Transformasi logaritma natural $\log(1 + \text{Harga})$ sebagai target pelatihan model regresi.
3. **`Merek_Model`:** Gabungan unik antara merek dan model/tipe spesifik mobil (`Model_Clean`) untuk stratifikasi distribusi tipe.

In [7]:
# 1. Usia Mobil
df['Usia_Mobil'] = 2025 - df['Tahun_Num']

# 2. Target Log-Transformed
df['Log_Harga'] = np.log1p(df['Harga_Num'])

# 3. Kunci Stratifikasi Merek + Tipe
df['Merek_Model'] = df['Merek_Clean'] + "_" + df['Model_Clean']

# Ringkasan kolom yang telah disiapkan
feature_cols = ['Merek_Clean', 'Model_Clean', 'Tahun_Num', 'Usia_Mobil', 'KM_Clean', 'Transmisi_Clean', 'Bahan_Bakar_Clean', 'Wilayah']
print(f"Total fitur yang disiapkan: {len(feature_cols)}")
df[feature_cols + ['Harga_Num', 'Log_Harga', 'Merek_Model']].head()


Total fitur yang disiapkan: 8


,Merek_Clean,Model_Clean,Tahun_Num,Usia_Mobil,KM_Clean,Transmisi_Clean,Bahan_Bakar_Clean,Wilayah,Harga_Num,Log_Harga,Merek_Model
0,Mercedes-Benz,A45 Amg,2014,11,47500.0,Automatic,Bensin,Jakarta,4.990000e+08,20.028117,Mercedes-Benz_A45 Amg
1,Mercedes-Benz,Sprinter,2021,4,22500.0,Manual,Diesel,Jakarta,1.850000e+09,21.338451,Mercedes-Benz_Sprinter
2,Mercedes-Benz,Glc200,2022,3,12500.0,Automatic,Bensin,Jakarta,7.200000e+08,20.394762,Mercedes-Benz_Glc200
3,Mercedes-Benz,Gle400,2016,9,47500.0,Automatic,Bensin,Jakarta,5.980000e+08,20.209101,Mercedes-Benz_Gle400
4,Mercedes-Benz,Lainnya,2010,15,72500.0,Automatic,Bensin,Luar Jabodetabek,1.590000e+08,18.884415,Mercedes-Benz_Lainnya


## 6. Pembagian Data: `StratifiedShuffleSplit` (80% Train : 20% Test)
Kita membagi dataset menjadi **80% Training** dan **20% Testing** menggunakan **`StratifiedShuffleSplit`** berdasarkan kombinasi `Merek_Model` (untuk tipe dengan $\ge 5$ iklan, dan `Merek_Clean_RARE` untuk varian langka):
- **Mengapa beralih ke `StratifiedShuffleSplit`?**  
  Karena fitur **tipe mobil (`Model_Clean`)** turut menjadi input utama prediksi (agar model membedakan *Toyota Agya* vs *Toyota Alphard* atau *Honda Brio* vs *Honda CR-V*), setiap tipe mobil perlu hadir secara proporsional di data training dan testing yang terpisah, sementara data uji tetap 100% diisolasi (*zero data leakage* antar baris iklan).

In [8]:
vc = df['Merek_Model'].value_counts()
strat_key = df['Merek_Model'].where(df['Merek_Model'].map(vc) >= 5, df['Merek_Clean'] + '_RARE')

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(sss.split(df, strat_key))

train_df = df.iloc[train_idx].copy().reset_index(drop=True)
test_df = df.iloc[test_idx].copy().reset_index(drop=True)

print(f"Total Data Bersih: {len(df)} baris")
print(f"Data Training: {len(train_df)} baris ({len(train_df)/len(df)*100:.1f}%)")
print(f"Data Testing : {len(test_df)} baris ({len(test_df)/len(df)*100:.1f}%)")
print(f"Jumlah tipe mobil (Model_Clean) unik di Train: {train_df['Model_Clean'].nunique()} | di Test: {test_df['Model_Clean'].nunique()}")

Total Data Bersih: 13359 baris
Data Training: 10687 baris (80.0%)
Data Testing : 2672 baris (20.0%)
Jumlah tipe mobil (Model_Clean) unik di Train: 211 | di Test: 162


## 7. Membangun Pipeline Preprocessing (Scikit-Learn ColumnTransformer)
Mempersiapkan transformer yang terintegrasi untuk standarisasi skala numerik dan *One-Hot Encoding* variabel kategorikal. Transformer ini di-fit **hanya** pada data training untuk menjamin kaidah machine learning yang benar.


In [9]:
num_features = ['Tahun_Num', 'Usia_Mobil', 'KM_Clean']
cat_features = ['Merek_Clean', 'Model_Clean', 'Transmisi_Clean', 'Bahan_Bakar_Clean', 'Wilayah']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ],
    remainder='drop'
)

# Fit preprocessor pada data training
X_train = train_df[num_features + cat_features]
y_train = train_df['Log_Harga']

X_test = test_df[num_features + cat_features]
y_test = test_df['Log_Harga']

preprocessor.fit(X_train)
print("ColumnTransformer berhasil di-fit pada data training.")

# Cek dimensi matriks fitur hasil transformasi
X_train_transformed = preprocessor.transform(X_train)
print(f"Bentuk matriks fitur setelah transformasi: {X_train_transformed.shape[0]} baris x {X_train_transformed.shape[1]} kolom")

ColumnTransformer berhasil di-fit pada data training.


Bentuk matriks fitur setelah transformasi: 10687 baris x 230 kolom


## 8. Ekspor Dataset Olahan & Artefak Preprocessing
Menyimpan dataset yang telah bersih dan siap pakai ke direktori `data/processed/`, serta menyimpan preprocessor ter-fit ke folder `models/`.


In [10]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('models', exist_ok=True)

proc_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
models_dir = '../models' if os.path.exists('../models') else 'models'

clean_path = os.path.join(proc_dir, 'clean_cars_data.csv')
df.to_csv(clean_path, index=False)
print(f"Data bersih disimpan ke: {clean_path}")

train_path = os.path.join(proc_dir, 'train_data.csv')
test_path = os.path.join(proc_dir, 'test_data.csv')
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)
print(f"Data train disimpan ke : {train_path}")
print(f"Data test disimpan ke  : {test_path}")

preproc_path = os.path.join(models_dir, 'preprocessor.joblib')
joblib.dump(preprocessor, preproc_path)
print(f"Preprocessor disimpan ke: {preproc_path}")

models_by_brand = {}
for brand, grp in df.groupby('Merek_Clean'):
    vc_b = grp['Model_Clean'].value_counts()
    models_by_brand[brand] = [{'name': str(m), 'count': int(c)} for m, c in vc_b.items()]

wilayah_counts = {str(k): int(v) for k, v in df['Wilayah'].value_counts().items()}

metadata = {
    'num_features': num_features,
    'cat_features': cat_features,
    'target_column': 'Harga_Num',
    'log_target_column': 'Log_Harga',
    'brands': TARGET_BRANDS,
    'models_by_brand': models_by_brand,
    'wilayah_list': sorted(df['Wilayah'].unique().tolist()),
    'wilayah_counts': wilayah_counts,
    'transmisi_list': sorted(df['Transmisi_Clean'].unique().tolist()),
    'bahan_bakar_list': sorted(df['Bahan_Bakar_Clean'].unique().tolist()),
    'total_clean_samples': int(len(df)),
    'train_samples': int(len(train_df)),
    'test_samples': int(len(test_df))
}

meta_path = os.path.join(models_dir, 'feature_metadata.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata fitur disimpan ke: {meta_path}")

Data bersih disimpan ke: ../data/processed\clean_cars_data.csv


Data train disimpan ke : ../data/processed\train_data.csv
Data test disimpan ke  : ../data/processed\test_data.csv
Preprocessor disimpan ke: ../models\preprocessor.joblib
Metadata fitur disimpan ke: ../models\feature_metadata.json


## 9. Ringkasan Hasil Preprocessing (Checklist Phase 3)

| Komponen Preprocessing | Status | Keterangan / Implementasi |
|---|---|---|
| **Data Cleaning** | ✅ Selesai | Filter 5 merek utama, unifikasi bensin/petrol, standarisasi seri Mercedes, perbaikan parsing KM `'K'`. |
| **Deduplikasi Data** | ✅ Selesai | Hapus duplikat `URL` & iklan repost identik hingga tersisa 13.359 iklan unik bebas ganda. |
| **Pembersihan Target** | ✅ Selesai | Drop NaN target, hapus outlier non-rasional (<25 Jt & >10 M). |
| **Imputasi Missing Values** | ✅ Selesai | Imputasi bertingkat pada Kilometer berbasis Model & Tahun kendaraan. |
| **Feature Engineering** | ✅ Selesai | Pembentukan `Usia_Mobil`, `Wilayah`, `Merek_Model`, `Model_Clean`, dan target `Log_Harga`. |
| **Validasi & Splitting** | ✅ Selesai | `StratifiedShuffleSplit` (80:20) berbasis merek/tipe dengan *0 leakage*. |
| **Pipeline Scikit-Learn** | ✅ Selesai | `ColumnTransformer` (`StandardScaler` + `OneHotEncoder` termasuk `Model_Clean`). |